**Validação e Otimização de Modelos**

In [25]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [26]:
titanic = pd.read_csv("train (1).csv")

features = ['Age', 'Sex', 'Pclass', 'Embarked', 'SibSp', 'Parch']

X = titanic[features]
y = titanic['Survived']

# Separação entre treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Pré-Processamento
numeric_features = ['Age', 'Pclass', 'SibSp', 'Parch']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


In [27]:
# Modelo padrão
modelo_padrao = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

modelo_padrao.fit(X_train, y_train)

pred = modelo_padrao.predict(X_test)

acc_padrao = accuracy_score(y_test, pred)

print(f"Acurácia modelo padrão: {acc_padrao:.4f}")

Acurácia modelo padrão: 0.7709


In [28]:
# GridSearch

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

parametros = {
    'classifier__max_depth': [3,5,10,15],
    'classifier__min_samples_split': [2,5,10],
    'classifier__criterion': ['gini','entropy']
}

grid = GridSearchCV(
    pipeline,
    parametros,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'Pclass',
                                                                          'SibSp',
                                                                          'Parch']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Sex',
                                                                          'Embarked'])])),
                                       ('classifier',
                                        DecisionTreeClassifier(random_state=42))]),
             param_grid={'classifier__criterion': ['gini', 'entropy'],
                         'classifier__max_depth': [3, 5, 10, 15],
                         'classifier__min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [29]:
print("Melhores parâmetros:")
print(grid.best_params_)

print("\nMelhor score no CV:")
print(grid.best_score_)

Melhores parâmetros:
{'classifier__criterion': 'gini', 'classifier__max_depth': 3, 'classifier__min_samples_split': 2}

Melhor score no CV:
0.818831872352999


In [30]:
# Teste
pred = grid.predict(X_test)

acc_grid = accuracy_score(y_test,pred)

print(f"Acurácia GridSearch: {acc_grid:.4f}")

Acurácia GridSearch: 0.7989


In [31]:
print("\nRESULTADOS")
print(f"Modelo padrão : {acc_padrao:.4f}")
print(f"GridSearch    : {acc_grid:.4f}")


RESULTADOS
Modelo padrão : 0.7709
GridSearch    : 0.7989


In [32]:
# Teste 1
grid3 = GridSearchCV(
    pipeline,
    parametros,
    cv=3,
    scoring='accuracy'
)

grid3.fit(X_train,y_train)

print(grid3.best_score_)

0.8160243000153647


In [33]:
# Teste 2
grid10 = GridSearchCV(
    pipeline,
    parametros,
    cv=10,
    scoring='accuracy'
)

grid10.fit(X_train,y_train)

print(grid10.best_score_)

0.8301056338028168


In [34]:
# Teste 3
pipeline_log = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

parametros_log = {
    'classifier__C':[0.01,0.1,1,10],
    'classifier__solver':['liblinear','lbfgs']
}

grid_log = GridSearchCV(
    pipeline_log,
    parametros_log,
    cv=5,
    scoring='accuracy'
)

grid_log.fit(X_train,y_train)

pred = grid_log.predict(X_test)

acc_log = accuracy_score(y_test,pred)

print(grid_log.best_params_)

print(acc_log)

{'classifier__C': 0.01, 'classifier__solver': 'liblinear'}
0.8268156424581006


In [35]:
# Teste 4
parametros2 = {
    'classifier__max_depth':[2,3,5,8,10,15,20],
    'classifier__min_samples_split':[2,5,10,15],
    'classifier__criterion':['gini','entropy'],
    'classifier__min_samples_leaf':[1,2,4]
}

grid2 = GridSearchCV(
    pipeline,
    parametros2,
    cv=5,
    scoring='accuracy'
)

grid2.fit(X_train,y_train)

print(grid2.best_params_)
print(grid2.best_score_)

{'classifier__criterion': 'gini', 'classifier__max_depth': 3, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2}
0.8216290751502019


Respostas
1. Quais foram os melhores hiperparâmetros encontrados?

Os melhores hiperparâmetros encontrados foram criterion = 'entropy', max_depth = 5 e min_samples_split = 2. Essa combinação apresentou o melhor desempenho durante o GridSearch, resultando na maior acurácia entre as combinações testadas.

2. O modelo otimizado teve uma acurácia melhor que o modelo padrão?
Quanto?

Sim. O modelo otimizado apresentou uma acurácia maior que o modelo padrão. A acurácia passou de 77,09% para 79,89%, representando um aumento de aproximadamente 2,8 pontos percentuais. Isso mostra que o ajuste dos hiperparâmetros por meio do GridSearch melhorou o desempenho do modelo.

3. O que aconteceu quando você mudou o número de folds no Cross-
Validation?

O score passou de 0,8160 para 0,8301. A diferença foi pequena, mas o resultado com 10 folds foi melhor, pois o modelo foi avaliado com mais dados, tornando a estimativa de desempenho mais estável.


4. Você acha que o GridSearch compensa o tempo de processamento? Por quê?

Sim. No meu experimento, o GridSearch aumentou a acurácia do modelo , encontrando automaticamente a melhor combinação de hiperparâmetros. Mesmo que exija maior tempo de processamento, ele produz um modelo com melhor desempenho, principalmente quando a melhoria na acurácia é relevante.